In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Visualize missing values
plt.figure(figsize=(12, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Analyze numeric columns
print(train_data[numeric_cols].describe())

# Analyze categorical columns
print(train_data[categorical_cols].describe())

# Check for anomalies in numeric columns
for col in numeric_cols:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=train_data[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# Check for anomalies in categorical columns
for col in categorical_cols:
    plt.figure(figsize=(12, 6))
    sns.countplot(y=train_data[col], order=train_data[col].value_counts().index)
    plt.title(f'Countplot of {col}')
    plt.show()


        Category  DayOfWeek PdDistrict           X          Y
0  LARCENY/THEFT  Wednesday   SOUTHERN -122.388380  37.783310
1   NON-CRIMINAL   Saturday   SOUTHERN -122.403405  37.775421
2  LARCENY/THEFT  Wednesday   NORTHERN -122.419581  37.789214
3  VEHICLE THEFT     Friday    BAYVIEW -122.389744  37.757909
4        ASSAULT     Friday    TARAVAL -122.478377  37.742877
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70244 entries, 0 to 70243
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Category    70244 non-null  object 
 1   DayOfWeek   70244 non-null  object 
 2   PdDistrict  70244 non-null  object 
 3   X           70244 non-null  float64
 4   Y           70244 non-null  float64
dtypes: float64(2), object(3)
memory usage: 2.7+ MB
None
Category      0
DayOfWeek     0
PdDistrict    0
X             0
Y             0
dtype: int64


Numeric Columns: Index(['X', 'Y'], dtype='object')
Categorical Columns: Index(['Category', 'DayOfWeek', 'PdDistrict'], dtype='object')
                  X             Y
count  70244.000000  70244.000000
mean    -122.422808     37.767867
std        0.026338      0.198551
min     -122.513642     37.707944
25%     -122.432915     37.752590
50%     -122.416452     37.775421
75%     -122.406992     37.784400
max     -120.500000     90.000000
             Category DayOfWeek PdDistrict
count           70244     70244      70244
unique             37         7         10
top     LARCENY/THEFT    Friday   SOUTHERN
freq            13951     10653      12539


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:17:27.539 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Category', 'DayOfWeek', 'PdDistrict'], 'Numeric': ['X', 'Y'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the training data
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/crime_category/train.csv')

# Separate features and target
X_train = train_data.drop('Category', axis=1)
y_train = train_data['Category']

# Handle missing values
fill_missing = FillMissingValue(features=['X', 'Y'], strategy='mean')
X_train = fill_missing.fit_transform(X_train)

# Encode categorical variables
label_encode = LabelEncode(features=['DayOfWeek', 'PdDistrict'])
X_train = label_encode.fit_transform(X_train)

# Normalize numerical features
standard_scale = StandardScale(features=['X', 'Y'])
X_train = standard_scale.fit_transform(X_train)

# Display the preprocessed training data
print(X_train.head())
print(y_train.head())


   DayOfWeek  PdDistrict         X         Y
0          6           7  1.307195  0.077780
1          2           7  0.736721  0.038043
2          6           4  0.122545  0.107514
3          0           0  1.255416 -0.050156
4          0           8 -2.109862 -0.125866
0    LARCENY/THEFT
1     NON-CRIMINAL
2    LARCENY/THEFT
3    VEHICLE THEFT
4          ASSAULT
Name: Category, dtype: object


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(X_train)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['DayOfWeek', 'PdDistrict', 'X', 'Y'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

# Assuming X_train and y_train are already defined from previous tasks
# X_train = ...
# y_train = ...

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Initialize and fit the XGBoost classifier
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_train)),
    learning_rate=0.1,
    max_depth=6,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

# Predict probabilities on the validation set
y_val_pred_proba = xgb.predict_proba(X_val)

# Calculate log loss
logloss = log_loss(y_val, y_val_pred_proba, labels=np.unique(y_train))
print(f'Validation Log Loss: {logloss}')


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36], got ['ARSON' 'ASSAULT' 'BAD CHECKS' 'BRIBERY' 'BURGLARY' 'DISORDERLY CONDUCT'
 'DRIVING UNDER THE INFLUENCE' 'DRUG/NARCOTIC' 'DRUNKENNESS'
 'EMBEZZLEMENT' 'EXTORTION' 'FAMILY OFFENSES' 'FORGERY/COUNTERFEITING'
 'FRAUD' 'GAMBLING' 'KIDNAPPING' 'LARCENY/THEFT' 'LIQUOR LAWS' 'LOITERING'
 'MISSING PERSON' 'NON-CRIMINAL' 'OTHER OFFENSES' 'PROSTITUTION'
 'RECOVERED VEHICLE' 'ROBBERY' 'RUNAWAY' 'SECONDARY CODES'
 'SEX OFFENSES FORCIBLE' 'SEX OFFENSES NON FORCIBLE' 'STOLEN PROPERTY'
 'SUICIDE' 'SUSPICIOUS OCC' 'TRESPASS' 'VANDALISM' 'VEHICLE THEFT'
 'WARRANTS' 'WEAPON LAWS']